**German Credit Dataset**

# 02 - Fairness Groups Analysis

**Objectives**
- Identify and define sensitive attributes related to fairness 
- Specify the favorable and unfavorable label values in the target variable
- Define privileged and unprivileged (sensitive) groups to be used in fairness analysis

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings

from aif360.datasets import BinaryLabelDataset

## 1. Load Data

In [ ]:
file_path = '../data/processed/german_df_processed_1.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [ ]:
df

## 2. Identifying Sensitive Attr. and Privileged/Unprivileged Groups

In [ ]:
df_eng = df.copy()

### 2.1. marriage_status_sex

The column "marriage_status_sex" encodes both gender and marital status simultaneously, with the following mapping:
    1: "male divorced/separated"
    2: "female divorced/separated/married"
    3: "male single"
    4: "male married/widowed"
    5: "female single"
For simplicity, a new column "sex" will be created to indicate only the individual's gender.

The "marriage_status_sex" column will then be dropped.

In [ ]:
sex_map = {
    1: 1,  # male
    2: 2,  # female
    3: 1,  # male
    4: 1,  # male
    5: 2,  # female
}

df_eng['sex'] = df_eng['marriage_status_sex'].map(sex_map)
df_eng = df_eng.drop('marriage_status_sex', axis=1)

In [ ]:
counts_sex = df_eng['sex'].value_counts()
percent_sex = (counts_sex / len(df_eng)) * 100

print("Fairness Setting (Sex Distribution)")
print(f"Male (1): {counts_sex.get(1)} instances ({percent_sex.get(1):.1f}% of total)")
print(f"Female (2): {counts_sex.get(2)} instances ({percent_sex.get(2):.1f}% of total)")

Since 69% of the individuals in the dataset are male, males will be considered the privileged group and females the unprivileged group.

### 2.2. foreign_worker?

In [ ]:
counts_foreign = df_eng['foreign_worker?'].value_counts()
percent_foreign = (counts_foreign / len(df_eng)) * 100

print("Fairness Setting (Foreign Worker Distribution)")
print(f"Foreign (True): {counts_foreign.get(1)} instances ({percent_foreign.get(1):.1f}% of total)")
print(f"Local (False): {counts_foreign.get(0)} instances ({percent_foreign.get(0):.1f}% of total)")

Since 96% of the individuals in the dataset are foreign, foreigns will be considered the privileged group and locals the unprivileged group.

### 2.3. age

To facilitate fairness analysis based on age, we create a binned age category.  
The central 75% of the age distribution is defined as the privileged group (Group 1), while individuals whose ages fall into the lower 12.5% or upper 12.5% are classified as the unprivileged group (Group 2).  

This binning strategy is necessary to define clear age-based group boundaries for fairness experiments, as raw age is a continuous variable.  

The original "age" column will then be dropped and the categorical "age_cat" variable will me used as the sensitive attribute.

In [ ]:
# 12.5% + 75% + 12.5% = 100%
lower_bound = df_eng['age'].quantile(0.125)
upper_bound = df_eng['age'].quantile(0.875)

# Group 1 (Privileged): Within the central 75%
# Group 2 (Unprivileged): Outside this range (distribution tails)
df_eng['age_cat'] = np.where(
    (df_eng['age'] >= lower_bound) & (df_eng['age'] <= upper_bound), 
    1, 
    2
)

df_eng = df_eng.drop(columns=['age'])

In [ ]:
counts = df_eng['age_cat'].value_counts()
percent_priv = (counts.get(1, 0) / len(df_eng)) * 100

print("Fairness Setting (75% Coverage)")
print(f"The central group (75%) is between {lower_bound:.1f} and {upper_bound:.1f} years old.")
print(f"Privileged Group (1): {counts.get(1)} instances ({percent_priv:.1f}% of total)")
print(f"Unprivileged Group (2): {counts.get(2)} instances")
print(f"Privileged Age Range: [{lower_bound} to {upper_bound}]")

Since 77% of the individuals fall within the central age group (from 24.0 to 49.1 years old), individuals in this range will be considered the privileged group, while those outside this range (the youngest and oldest 12.5%) will be considered the unprivileged group.

## 3. Fairness Variable Definitions

In [ ]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Between 24.0 and 49.1 years old
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Other ages

## 4. Exporting Dataset

In [ ]:
file_out_path = '../data/processed'

df_eng.to_csv(file_out_path + '/german_df_processed_2.csv', index=False)